# Model Definition

[Prev](02_data.ipynb) | [Index](00_index.ipynb) | [Next](04_train.ipynb)

---


In [12]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# Custom Lightweight DenseNet-inspired Model
def create_custom_densenet(input_shape, num_classes, growth_rate=8, compression=0.5):
    inputs = tf.keras.Input(shape=input_shape)
    x = layers.BatchNormalization()(inputs)
    x = layers.Conv2D(growth_rate * 2, kernel_size=(3, 3), padding='same', activation='relu')(x)
    
    # Lightweight Dense Block
    def dense_block(x, num_layers, growth_rate):
        for _ in range(num_layers):
            out = layers.BatchNormalization()(x)
            out = layers.Conv2D(growth_rate, kernel_size=(3, 3), padding='same', activation='relu')(out)
            x = layers.Concatenate()([x, out])  # Concatenate input and output
        return x
    
    # Lightweight Transition Block
    def transition_block(x, compression):
        reduced_filters = int(x.shape[-1] * compression)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(reduced_filters, kernel_size=(1, 1), padding='same', activation='relu')(x)
        x = layers.AveragePooling2D(pool_size=(2, 2))(x)
        return x
    
    # Adding Dense Blocks and Transition Layers
    num_dense_layers = [3, 3, 3]  # Number of layers in each dense block
    for num_layers in num_dense_layers:
        x = dense_block(x, num_layers, growth_rate)
        x = transition_block(x, compression)
    
    # Classifier Head
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    return model

# Create Model
model = create_custom_densenet(input_shape=(64, 64, 3), num_classes=24)

# Compile Model with Gradient Clipping
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=1.0)  # Prevent gradient explosion
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])


W0000 00:00:1749898729.744163    6551 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
